# Colab runner: the full loss / stabilizer grid

Trains the 100 configurations of `swarm/experiments.py` `GRID` on a Colab GPU:

| Choice | Options |
|---|---|
| formation loss | chamfer, matching, sinkhorn, energy, laplacian |
| loss form | plain, robust (Huber for squared losses; smooth |x| for energy) |
| time weighting | final, last 20%, every step, running + terminal, discounted |
| gradient stabilizer | truncated BPTT (every 25 steps), temporal gradient decay (0.92) |

3,000 iterations each, no early stopping, geometric shapes, 8-96 drones. Results go to **Google Drive**, so a disconnect loses at most the runs in progress: re-run the cells and finished runs are skipped.

**Steps**: 1 runtime → 2 Drive + code → 3 timing test (~10 min, decide) → 4 run the grid → 5 progress → 6 score the runs.

Runtime → Change runtime type → **A100 GPU** (or L4/T4, slower). Keep this tab open while it runs.

## 1. Check the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

name, memory.total [MiB]
Tesla T4, 15360 MiB
8


## 2. Google Drive and the code

`OUT` is where every run is saved (on Drive). The code comes from GitHub (`BRANCH`); if the repo is private, use the ZIP option instead: zip the project folder, put it on Drive, and set `ZIP`.

In [6]:
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/drone-swarm-grid"      # results (persist across sessions)
REPO = "https://github.com/chrisdollo/838b-drone-swarm.git"
BRANCH = "main"
ZIP = ""                                             # e.g. "/content/drive/MyDrive/project-drone-swarm.zip" instead of GitHub

import os, subprocess
os.makedirs(OUT, exist_ok=True)
if not os.path.exists("/content/repo"):
    if ZIP:
        subprocess.run(["unzip", "-q", ZIP, "-d", "/content/zip"], check=True)
        inner = [d for d in os.listdir("/content/zip") if not d.startswith("__")]
        os.rename(f"/content/zip/{inner[0]}", "/content/repo")
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/repo"], check=True)
%cd /content/repo
!git log --oneline -1 2>/dev/null || true
!pip install -q hershey-fonts
!python -c "import torch, swarm.grid; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!ls data/benchmark_v3 | head -3

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/repo
4c6bf4d (HEAD -> main, origin/main, origin/HEAD) phase 2 implementation
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import torch, swarm.grid; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    ^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'swarm.grid'
ls: cannot access 'data/benchmark_v3': No such file or directory


## 3. Timing test (~10 minutes)

Times 40 full-length iterations with 1, 4 and 8 runs at once on this GPU and prints the estimated time for the whole grid. Pick the fastest `--jobs` for step 4. If the estimate is too long, run a subset with `ONLY` (e.g. `["robust"]` = 50 runs, or `["energy"]`).

In [3]:
!python -m swarm.grid bench --device cuda --jobs 1 4 8

/usr/bin/python3: No module named swarm.grid


## 4. Run the grid

Resumable: finished runs (those with `history.json` in `OUT`) are skipped, so after a disconnect just run this cell again. One line is printed per finished run with the time left. Per-run logs: `OUT/logs/`.

In [ ]:
JOBS = 8          # from the timing test
ONLY = []         # e.g. ["robust"], ["energy"], ["energy", "robust"]; [] = all 100
only = ("--only " + " ".join(ONLY)) if ONLY else ""
!python -m swarm.grid run --out "$OUT" --device cuda --jobs $JOBS $only

## 5. Progress (any time, also from a new session after step 2)

In [ ]:
!python -m swarm.grid list --out "$OUT"

## 6. Score the finished runs

Each finished run's best checkpoint on the fixed benchmark (training and held-out shapes, all sizes): slots filled, slot error, formed %, steps to form, shared slots, close pairs, small vs large swarms, plus training health (gradient norms, effort, collapse). Saved to `OUT/grid_results.json` after every run, so it can be interrupted. Then download that file (or the whole `OUT` folder) back to the repo's `results/`.

In [ ]:
!python -m swarm.grid summarize --out "$OUT" --device cuda
!python -m swarm.grid list --out "$OUT"